# P7513 sanity overlay

This notebook recreates the `P7513_sanity_overlay` plot from the visualization stage using the pregenerated P7513 MERSCOPE and Xenium SpatialData zarrs.

The plotting cell below copies the top-level code path from `plot_pair_sanity_crops` so the layout, crop size, assignment layer, output path, and panel calls can be edited interactively without rerunning the full Nextflow visualization stage.

## Kernel

Run this notebook with the MerXen environment that has `spatialdata` installed. The workflow conda environment at `work/conda/env-5ab9b2fa7b005685690639bfde24c694` imports the plotting stack successfully on this machine.

In [ ]:
from pathlib import Path
import sys
from typing import Any

import matplotlib.pyplot as plt
import spatialdata as sd
from matplotlib import patheffects as path_effects
from matplotlib.lines import Line2D

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "merxen").exists():
    REPO_ROOT = Path("/home/becalia/code/MerXen")

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from merxen.plotting import save_figure
from merxen.visualization.sanity_plots import (
    SANITY_ASSIGNMENT_SHAPE_KEY,
    SANITY_CROP_SIZE_UM,
    SANITY_MAX_TRANSCRIPTS_PER_PANEL,
    SANITY_RANDOM_STATE,
    SANITY_SCALE_BAR_UM,
    _add_scale_bar,
    _build_pair_sanity_plans,
    _choose_crop_bbox,
    _crop_points,
    _crop_single_shape,
    _get_background_image_crop,
    _ordered_legend_handles,
    _ordered_sanity_shape_keys,
    _reference_shape_key,
    _sanity_shape_style,
    _sanity_shape_zorder,
    plot_pair_crop_location,
)


In [ ]:
PAIR_ID = "P7513"

MERSCOPE_ZARR_PATH = REPO_ROOT / "results" / PAIR_ID / "merscope" / "latest" / "latest_spatialdata.zarr"
XENIUM_ZARR_PATH = REPO_ROOT / "results" / PAIR_ID / "xenium" / "latest" / "latest_spatialdata.zarr"

OUTPUT_DIR = REPO_ROOT / "results" / PAIR_ID / "visualization" / "visualize_out"
OUTPUT_PATH = OUTPUT_DIR / f"{PAIR_ID}_sanity_overlay_interactive.png"
OUTPUT_PDF_PATH = OUTPUT_PATH.with_suffix(".pdf")

print(f"MERSCOPE: {MERSCOPE_ZARR_PATH}")
print(f"Xenium:   {XENIUM_ZARR_PATH}")
print(f"PNG output: {OUTPUT_PATH}")
print(f"PDF output: {OUTPUT_PDF_PATH}")


In [ ]:
merscope_sdata = sd.read_zarr(MERSCOPE_ZARR_PATH)
xenium_sdata = sd.read_zarr(XENIUM_ZARR_PATH)

print("MERSCOPE layers")
print(f"  images: {list(merscope_sdata.images.keys())}")
print(f"  shapes: {list(merscope_sdata.shapes.keys())}")
print(f"  points: {list(merscope_sdata.points.keys())}")

print("Xenium layers")
print(f"  images: {list(xenium_sdata.images.keys())}")
print(f"  shapes: {list(xenium_sdata.shapes.keys())}")
print(f"  points: {list(xenium_sdata.points.keys())}")


In [ ]:
def _interactive_sanity_shape_style(
    shape_key: str,
    *,
    proseg_color: str,
    cellpose_color: str,
    original_segmentation_color: str,
) -> tuple[str, str]:
    """Notebook-local colors for the publication-style MERSCOPE crop."""
    label, color = _sanity_shape_style(shape_key)
    canonical_key = str(shape_key).removesuffix("_aligned_nonrigid")
    if canonical_key == "MOSAIK_proseg":
        return label, proseg_color
    if canonical_key == "MOSAIK_cellpose":
        return label, cellpose_color
    if canonical_key in {"merscope_cell_boundaries", "xenium_cell_boundaries"}:
        return label, original_segmentation_color
    return label, color


def _interactive_legend_label(label: str) -> str:
    """Normalize notebook legend labels for the final figure."""
    if label == "Original segmentation":
        return "Original Seg"
    if label in {"Assigned tx", "Unassigned tx"}:
        return "Transcripts"
    return label


def _order_interactive_legend_handles(handles: list[Line2D]) -> list[Line2D]:
    """Sort legend entries in the requested figure order."""
    order = {
        "Transcripts": 0,
        "Original Seg": 1,
        "Cellpose-SAM": 2,
        "ProSeg": 3,
    }
    return sorted(handles, key=lambda handle: order.get(str(handle.get_label()), 99))


def _add_interactive_scale_bar(
    ax: plt.Axes,
    bbox: tuple[float, float, float, float],
    *,
    length_um: float = 20.0,
    linewidth: float = 10.0,
    fontsize: float = 20.0,
) -> None:
    """Draw a prominent scale bar in the bottom-right corner of a crop."""
    x0, y0, x1, y1 = bbox
    x_span = x1 - x0
    y_span = y1 - y0
    if x_span <= 0 or y_span <= 0:
        return

    shown_length = min(length_um, x_span * 0.8)
    pad_x = x_span * 0.06
    pad_y = y_span * 0.07
    x_end = x1 - pad_x
    x_start = x_end - shown_length
    y = y0 + pad_y
    text_effects = [
        path_effects.Stroke(linewidth=3.0, foreground="black"),
        path_effects.Normal(),
    ]
    line_effects = [
        path_effects.Stroke(linewidth=linewidth + 4.0, foreground="black"),
        path_effects.Normal(),
    ]
    ax.plot(
        [x_start, x_end],
        [y, y],
        color="white",
        linewidth=linewidth,
        solid_capstyle="butt",
        path_effects=line_effects,
        zorder=20,
    )
    ax.text(
        0.5 * (x_start + x_end),
        y + y_span * 0.035,
        f"{shown_length:.0f} µm",
        color="white",
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold",
        path_effects=text_effects,
        zorder=20,
    )


def plot_sanity_crop_panel(
    ax: plt.Axes,
    sdata_obj: Any,
    dataset_name: str,
    *,
    crop_size_um: float = SANITY_CROP_SIZE_UM,
    crop_bbox: tuple[float, float, float, float] | None = None,
    center_xy: tuple[float, float] | None = None,
    assignment_shape_key: str | None = SANITY_ASSIGNMENT_SHAPE_KEY,
    prefer_aligned_vectors: bool = True,
    zarr_path: Path | str | None = None,
    show_axes: bool = False,
    show_grid: bool = False,
    assigned_transcript_color: str = "yellow",
    boundary_linewidth: float = 1.5,
    cellpose_color: str = "#C084FC",
    legend_fontsize: float = 16.0,
    legend_linewidth: float = 4.0,
    legend_marker_size: float = 9.0,
    original_segmentation_color: str = "#ff7f0e",
    original_segmentation_linewidth: float | None = None,
    proseg_color: str = "#8BC34A",
    scale_bar_fontsize: float = 20.0,
    scale_bar_linewidth: float = 10.0,
    scale_bar_um: float | None = 20.0,
    transcript_point_alpha: float = 0.75,
    transcript_point_size: float = 18.0,
    unassigned_transcript_color: str = "#9CA3AF",
) -> None:
    """Draw one MOSAIK-style image, shape, and assignment sanity crop panel."""
    if crop_bbox is None:
        bbox, ref_shape_key = _choose_crop_bbox(
            sdata_obj,
            size_um=crop_size_um,
            center_xy=center_xy,
            prefer_aligned=prefer_aligned_vectors,
        )
    else:
        bbox = crop_bbox
        ref_shape_key = _reference_shape_key(
            sdata_obj,
            prefer_aligned=prefer_aligned_vectors,
        )
    x0, y0, x1, y1 = bbox
    bg = _get_background_image_crop(sdata_obj, dataset_name, bbox, zarr_path=zarr_path)
    if bg is not None:
        ax.imshow(
            bg["rgb"],
            extent=bg["extent_um"],
            origin="lower",
            interpolation="nearest",
            alpha=0.95,
        )

    shape_keys = _ordered_sanity_shape_keys(
        sdata_obj,
        dataset_name=dataset_name,
        prefer_aligned=prefer_aligned_vectors,
    )
    shape_handles = []
    legend_labels_seen: set[str] = set()

    for shape_key in shape_keys:
        label, color = _interactive_sanity_shape_style(
            shape_key,
            proseg_color=proseg_color,
            cellpose_color=cellpose_color,
            original_segmentation_color=original_segmentation_color,
        )
        canonical_shape_key = str(shape_key).removesuffix("_aligned_nonrigid")
        line_width = boundary_linewidth
        if (
            canonical_shape_key in {"merscope_cell_boundaries", "xenium_cell_boundaries"}
            and original_segmentation_linewidth is not None
        ):
            line_width = original_segmentation_linewidth

        try:
            shp_crop = _crop_single_shape(sdata_obj, shape_key, bbox)
        except Exception as exc:  # noqa: BLE001
            print(
                f"[{dataset_name}] Warning: failed to crop shapes[{shape_key}] ({exc})"
            )
            continue

        if len(shp_crop) == 0:
            continue

        shp_crop.boundary.plot(
            ax=ax,
            linewidth=line_width,
            color=color,
            alpha=0.95,
            zorder=_sanity_shape_zorder(shape_key),
        )
        display_label = _interactive_legend_label(label)
        if display_label in legend_labels_seen:
            continue
        legend_labels_seen.add(display_label)
        shape_handles.append(
            Line2D(
                [0],
                [0],
                color=color,
                lw=legend_linewidth,
                label=display_label,
            )
        )

    tx_crop, _points_key, _assign_col = _crop_points(
        sdata_obj,
        bbox,
        max_points=SANITY_MAX_TRANSCRIPTS_PER_PANEL,
        random_state=SANITY_RANDOM_STATE,
        assignment_shape_key=assignment_shape_key,
        prefer_aligned_points=prefer_aligned_vectors,
        prefer_aligned_assignment=prefer_aligned_vectors,
    )

    tx_handles = []
    if len(tx_crop) > 0:
        unassigned = tx_crop[~tx_crop["assigned"]]
        assigned = tx_crop[tx_crop["assigned"]]
        combined_transcript_legend = (
            str(unassigned_transcript_color).lower()
            == str(assigned_transcript_color).lower()
        )

        if len(unassigned) > 0:
            ax.scatter(
                unassigned["x_um"],
                unassigned["y_um"],
                s=transcript_point_size,
                c=unassigned_transcript_color,
                alpha=transcript_point_alpha,
                rasterized=True,
            )
            if not combined_transcript_legend:
                tx_handles.append(
                    Line2D(
                        [0],
                        [0],
                        marker="o",
                        linestyle="None",
                        color=unassigned_transcript_color,
                        label="Unassigned tx",
                        markersize=legend_marker_size,
                    )
                )

        if len(assigned) > 0:
            ax.scatter(
                assigned["x_um"],
                assigned["y_um"],
                s=transcript_point_size,
                c=assigned_transcript_color,
                alpha=transcript_point_alpha,
                rasterized=True,
            )
            if not combined_transcript_legend:
                tx_handles.append(
                    Line2D(
                        [0],
                        [0],
                        marker="o",
                        linestyle="None",
                        color=assigned_transcript_color,
                        label="Assigned tx",
                        markersize=legend_marker_size,
                    )
                )

        if combined_transcript_legend:
            tx_handles.append(
                Line2D(
                    [0],
                    [0],
                    marker="o",
                    linestyle="None",
                    color=assigned_transcript_color,
                    label="Transcripts",
                    markersize=legend_marker_size,
                )
            )

    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect("equal")
    if show_axes or show_grid:
        ax.set_xlabel("x (um)")
        ax.set_ylabel("y (um)")
        ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
        ax.grid(show_grid, color="white", linewidth=0.6, alpha=0.45)
    else:
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.set_title("")
    if scale_bar_um is not None:
        _add_interactive_scale_bar(
            ax,
            bbox,
            length_um=scale_bar_um,
            linewidth=scale_bar_linewidth,
            fontsize=scale_bar_fontsize,
        )

    handles = tx_handles + shape_handles
    if handles:
        handles = _order_interactive_legend_handles(handles)
        legend = ax.legend(
            handles=handles,
            loc="lower left",
            frameon=True,
            facecolor="white",
            edgecolor="black",
            framealpha=1.0,
            fontsize=legend_fontsize,
        )
        legend.set_zorder(1_000)
        legend.get_frame().set_alpha(1.0)

    if ref_shape_key not in shape_keys:
        print(f"[{dataset_name}] Warning: reference shape {ref_shape_key} not plotted")

In [ ]:
def _manual_bbox_from_limits(
    manual_xlim: tuple[float, float] | None,
    manual_ylim: tuple[float, float] | None,
) -> tuple[float, float, float, float] | None:
    """Convert visible x/y limits into the internal (xmin, ymin, xmax, ymax) bbox."""
    if manual_xlim is None and manual_ylim is None:
        return None
    if manual_xlim is None or manual_ylim is None:
        raise ValueError("Set both manual_xlim=(xmin, xmax) and manual_ylim=(ymin, ymax).")

    xmin, xmax = sorted(map(float, manual_xlim))
    ymin, ymax = sorted(map(float, manual_ylim))
    if xmin == xmax or ymin == ymax:
        raise ValueError("Manual crop limits must span a non-zero area.")
    return (xmin, ymin, xmax, ymax)


def make_pair_sanity_overlay(
    merscope_sdata,
    xenium_sdata,
    *,
    merscope_zarr_path,
    xenium_zarr_path,
    manual_xlim: tuple[float, float] | None = None,
    manual_ylim: tuple[float, float] | None = None,
    crop_size_um=SANITY_CROP_SIZE_UM,
    assignment_shape_key=SANITY_ASSIGNMENT_SHAPE_KEY,
    figure_size=(8, 8),
    show_axes=False,
    show_grid=False,
    assigned_transcript_color="yellow",
    boundary_linewidth=1.5,
    cellpose_color="#C084FC",
    legend_fontsize=16.0,
    legend_linewidth=4.0,
    legend_marker_size=9.0,
    original_segmentation_color="#ff7f0e",
    original_segmentation_linewidth=None,
    proseg_color="#8BC34A",
    scale_bar_fontsize=20.0,
    scale_bar_linewidth=10.0,
    scale_bar_um=20.0,
    transcript_point_alpha=0.75,
    transcript_point_size=18.0,
    unassigned_transcript_color="#9CA3AF",
):
    """Interactive copy of `plot_pair_sanity_crops` that draws MERSCOPE only."""
    merscope_plan, xenium_plan = _build_pair_sanity_plans(
        merscope_sdata,
        xenium_sdata,
        crop_size_um=crop_size_um,
    )
    manual_bbox = _manual_bbox_from_limits(manual_xlim, manual_ylim)
    merscope_bbox = manual_bbox if manual_bbox is not None else merscope_plan.display_bbox

    fig, ax = plt.subplots(1, 1, figsize=figure_size, constrained_layout=True)
    plot_sanity_crop_panel(
        ax,
        merscope_sdata,
        "MERSCOPE",
        crop_bbox=merscope_bbox,
        crop_size_um=crop_size_um,
        assignment_shape_key=assignment_shape_key,
        prefer_aligned_vectors=merscope_plan.prefer_aligned_vectors,
        zarr_path=merscope_zarr_path,
        show_axes=show_axes,
        show_grid=show_grid,
        assigned_transcript_color=assigned_transcript_color,
        boundary_linewidth=boundary_linewidth,
        cellpose_color=cellpose_color,
        legend_fontsize=legend_fontsize,
        legend_linewidth=legend_linewidth,
        legend_marker_size=legend_marker_size,
        original_segmentation_color=original_segmentation_color,
        original_segmentation_linewidth=original_segmentation_linewidth,
        proseg_color=proseg_color,
        scale_bar_fontsize=scale_bar_fontsize,
        scale_bar_linewidth=scale_bar_linewidth,
        scale_bar_um=scale_bar_um,
        transcript_point_alpha=transcript_point_alpha,
        transcript_point_size=transcript_point_size,
        unassigned_transcript_color=unassigned_transcript_color,
    )
    return fig, merscope_plan, xenium_plan


In [ ]:
fig, merscope_plan, xenium_plan = make_pair_sanity_overlay(
    merscope_sdata,
    xenium_sdata,
    merscope_zarr_path=MERSCOPE_ZARR_PATH,
    xenium_zarr_path=XENIUM_ZARR_PATH,
    # Set these to crop the MERSCOPE panel to the visible coordinate window.
    manual_xlim=(5500, 5560),
    manual_ylim=(5230, 5290),
    show_axes=False,
    show_grid=False,
    assigned_transcript_color="gold",
    boundary_linewidth=3.5,
    cellpose_color="mediumorchid",
    legend_fontsize=26,
    legend_linewidth=10,
    legend_marker_size=12,
    original_segmentation_color="tomato",
    original_segmentation_linewidth=5,
    proseg_color="aquamarine",
    scale_bar_fontsize=30,
    scale_bar_linewidth=15,
    scale_bar_um=20,
    transcript_point_alpha=0.9,
    transcript_point_size=18,
    unassigned_transcript_color="gold",
)

In [ ]:
# Optional: save the interactive output and matching crop-location helper.
# save_figure() is the repo helper used by production plots; it writes PNG plus a sibling PDF.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_figure(fig, OUTPUT_PATH, dpi=220)

crop_location_path = OUTPUT_PATH.with_name(
    f"{OUTPUT_PATH.stem}_crop_location{OUTPUT_PATH.suffix}"
)
crop_location_pdf_path = crop_location_path.with_suffix(".pdf")
plot_pair_crop_location(
    merscope_sdata,
    xenium_sdata,
    crop_location_path,
    merscope_plan=merscope_plan,
    xenium_plan=xenium_plan,
)

print(f"Saved PNG: {OUTPUT_PATH}")
print(f"Saved PDF: {OUTPUT_PDF_PATH}")
print(f"Saved crop-location PNG: {crop_location_path}")
print(f"Saved crop-location PDF: {crop_location_pdf_path}")
